# 01: End-to-end Data Preparation

This notebook demostrate end-to-end data prearation workflow including:
- `world-grid` generation
- data harmonization
- data ingestion
- data preparatioin -> feed to a session

## Environment Setup & Imports

> **Prerequisite**: Ensure this notebook is attached to a cluster with the `landseg` wheel pre-installed (e.g. Databricks compute via cluster init script or cluster libraries) or an active local virtual environment. No in-notebook pip installation is required.

In [ ]:
# standard imports
import json
import pathlib
import sys
# third-party imports
import numpy
# local imports
# Attempt importing landseg directly from site-packages (wheel install).
# Fall back to local '../src' only when running inside the repository clone.
try:
    import landseg.adapters.api as api
    import landseg.mlops as mlops
    import landseg.notebooks as nb
    import landseg.testing as testing
except ImportError:
    repo_src = pathlib.Path.cwd().parent / 'src'
    if repo_src.exists():
        sys.path.insert(0, str(repo_src))
    import landseg.adapters.api as api
    import landseg.mlops as mlops
    import landseg.notebooks as nb
    import landseg.testing as testing

# Resolve experiment root path (auto-bridges Databricks widgets, env vars, or local defaults).
EXP_ROOT = mlops.resolve_exp_root()
print(f'Successfully imported landseg and dependencies! Active experiment root: {EXP_ROOT}')

## Generate Dummy Dataset


For testing purposes we can use the shipped utility to generate a set of real rasters (`.tif` files) to demo the following process

In [ ]:
testing.generate_pre_harmonization_data(f'{EXP_ROOT}/input')

## Configure World Grid and Run

The **World Grid** defines the canonical spatial reference frame (CRS, resolution, tile dimensions, stride, and extent) used throughout data harmonization, ingestion, and inference.


In [ ]:
# NOTE
# Configure the spatial grid specifications on the WorldGridConfigurator.
# The world grid will be materialized and persisted in artifacts/world_grids/.

# declare widgets for interactive grid parameterization
widgets = nb.NotebookWidgets()
crs = widgets.dropdown('crs', default='EPSG:3161', choices=['EPSG:3161', 'EPSG:4326', 'EPSG:3857'], label='Target CRS')
tile_size = widgets.int('tile_size', default=256, label='Tile Size (px)')
tile_stride = widgets.int('tile_stride', default=128, label='Tile Stride (px)')

# init world grid configurator
grid_configurator = api.WorldGridConfigurator(EXP_ROOT)

# set target spatial grid (CRS, tile size, stride, and extent reference)
grid_configurator.set_grid(
    crs=crs,
    tile_size=tile_size,
    tile_stride=tile_stride,
    reference_raster=str(pathlib.Path(EXP_ROOT) / 'input/reference_raster/sample_extent.tif'),
)

# run pre-flight readiness check for world grid
grid_configurator.preflight()

# run world-grid pipeline
api.run(grid_configurator.running_root_config)


## Configure Data Harmonization (ETL) and Run


In [ ]:
# NOTE
# Data harmonization reprojects and stacks raw rasters into canonical Virtual Rasters (.vrt)
# aligned directly to the spatial world grid generated above.

# init harmonization configurator
harmonize_configurator = api.DataHarmonizationConfigurator(EXP_ROOT)

# set dataset manifest containing metadata for all input rasters
harmonize_configurator.set_dataset_manifest(
    dataset_manifest=EXP_ROOT / 'input/raw_data/manifest.json'
)

# set resampling algorithms for continuous and categorical rasters
harmonize_configurator.set_resampling(
    continuous='bilinear',
    categorical='nearest'
)

# set target spatial world grid tile dimensions (matches world grid generated above)
harmonize_configurator.set_grid(
    tile_size=256,
    tile_stride=128
)

# run pre-flight readiness check for data harmonization
harmonize_configurator.preflight()

# run data-harmonize pipeline
api.run(harmonize_configurator.running_root_config)


## Configure Data Ingestion and Run

Data ingestion slices harmonized rasters into canonical data blocks (`.npz`) and manages collision policies against the block pool.

- **Atomic Single-Batch Ingestion**: Use `DataIngestionConfigurator` to process an explicit single harmonization batch run.
- **Multi-Batch Catch-Up Workflow**: Use `BatchIngestConfigurator` to discover and sequentially ingest all pending batches in `harmonization_runs.json`.


In [ ]:
# NOTE
# Configure ingestion workflow using BatchIngestConfigurator
# (or use DataIngestionConfigurator for atomic single-batch target execution).

# init configurator for data ingestion workflow
ingest_configurator = api.DataIngestionConfigurator(EXP_ROOT)

# set whether to force rebuild ingestion data artifacts (default: False)
ingest_configurator.set_rebuild(False)

# set collision policy ('skip', 'overwrite', or 'error')
ingest_configurator.set_collision_policy('skip')

# optionally configure feature engineering (e.g. slope, ndvi)
# configurator.set_feature_engineering(add_topo=['slope'], add_spectral=['ndvi'])

# run pre-flight readiness check for batch ingestion
ingest_configurator.preflight()

# run batch ingestion workflow
api.run(ingest_configurator.running_root_config)


## Configure Data Preparation and Run

Data preparation is the intermediate step between data ingestion and model training. It:
1. **Selects input feature channels** using named schemes (e.g. `'rgb_nir'`, `'surface'`) or inline band lists.
2. **Resolves target reclassification schemes** (e.g. `'binary'` vs canonical `'raw'` classes) into multi-head targets.
3. **Partitions data blocks** into training, validation, and holdout test splits.
4. **Calculates per-band statistics** and normalizes images and label stacks into prepared `.npz` block artifacts.

In [ ]:
# init configurator
prep_configurator = api.DataPreparationConfigurator(EXP_ROOT)

# 1. Feature Channel Selection (Optional)
# Select named schemes defined in sidecars or provide inline band lists
prep_configurator.set_features({
    'sentinel2': 'rgb_nir',  # Use blue, green, red, and nir bands from Sentinel-2
})

# 2. Target Reclassification & Multi-Head Topology (Optional)
# Select named schemes or keep base raw classes
prep_configurator.set_targets({
    'landcover': 'binary',   # Reclassify into binary forest / water hierarchy
    'leadspc': 'raw',        # Keep canonical raw species classes
})

# 3. Set partition ratios (e.g. 20% validation, remaining for training)
prep_configurator.set_partition(
    validation_blocks_ratio=0.20,
    test_holdout_blocks_ratio=0.00
)

# 4. Set rebuild flag
prep_configurator.set_rebuild(True)

# run pre-flight readiness check for data preparation
prep_configurator.preflight()

# run data preparation pipeline
api.run(prep_configurator.running_root_config)


## Inspect Prepared Artifacts & Dataset Schema

Let's inspect the compiled `schema.json` and a sample prepared `.npz` data block:

In [ ]:
# 1. Read Dataset Transform Schema
schema_path = EXP_ROOT / 'artifacts/prepared_data/schema.json'
with open(schema_path, 'r', encoding='UTF-8') as f:
    schema = json.load(f)

print("=== Dataset Transform Schema ===")
print(f"Train Blocks: {len(schema['train_blocks'])}")
print(f"Val Blocks:   {len(schema['val_blocks'])}")
print(f"Test Blocks:  {len(schema['test_blocks'])}")
print("\nImage Channels Normalization Stats (Mean / Std):")
for band, stats in schema['image_stats'].items():
    print(f"  {band}: mean={stats['current_mean']:.4f}, std={stats['std']:.4f}")

# 2. Inspect a sample prepared block
_, sample_block_path = next(iter(schema['train_blocks'].items()))
block_data = numpy.load(sample_block_path)

print(f"\n=== Sample Block ({sample_block_path.name}) ===")
print(f"Keys: {list(block_data.keys())}")
print(f"Image Array Shape (C, H, W): {block_data['image'].shape}")
print(f"Label Array Shape (L, H, W): {block_data['label'].shape}")
print(f"Unique Label Values in Base Head: {numpy.unique(block_data['label'][0])}")